[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/pytorch-lab/blob/main/04_early_stopping.ipynb)

# 04 - Ранняя остановка и checkpointing

**Цель.** Добавить к обучению регрессионной модели раннюю остановку и сохранение лучших весов на диск.

**Результаты обучения:**
- создавать простой класс EarlyStopping
- сохранять лучшие веса через torch.save
- восстанавливать checkpoint через torch.load
- сравнивать влияние разных значений patience
- отмечать момент остановки на графике loss

## Источники
- [Heaton: Early stopping](https://github.com/jeffheaton/app_deep_learning/blob/main/t81_558_class_04_2_early_stop.ipynb)

Импортируем библиотеки. Ноутбук самостоятельный: ниже есть функция загрузки и подготовки Auto MPG.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

Функция `load_mpg` повторяет pipeline из ноутбука 03: загрузка, очистка, one-hot кодирование, train/validation split и стандартизация.

In [ ]:
def load_mpg():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
    columns = [
        "mpg", "cylinders", "displacement", "horsepower", "weight",
        "acceleration", "model_year", "origin", "car_name",
    ]
    df = pd.read_csv(
        url,
        names=columns,
        sep=r"\s+",
        na_values="?",
        comment="\t",
    )
    df = df.dropna().reset_index(drop=True)
    y = df["mpg"].astype("float32").to_numpy().reshape(-1, 1)
    X = df.drop(columns=["mpg", "car_name"])
    X = pd.get_dummies(X, columns=["origin"], prefix="origin", dtype=float)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype("float32")
    X_val_scaled = scaler.transform(X_val).astype("float32")

    train_ds = TensorDataset(torch.from_numpy(X_train_scaled), torch.from_numpy(y_train))
    val_ds = TensorDataset(torch.from_numpy(X_val_scaled), torch.from_numpy(y_val))
    return train_ds, val_ds, X_train.columns.tolist(), scaler

Опишем ту же регрессионную архитектуру, что и раньше.

In [ ]:
class MPGRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 25),
            nn.ReLU(),
            nn.Linear(25, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x):
        return self.net(x)

Класс `EarlyStopping` отслеживает лучший validation loss. При улучшении сохраняет веса, иначе увеличивает счетчик; когда счетчик достигает `patience`, обучение останавливается.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, path="best_model.pt", min_delta=0.0):
        self.patience = patience
        self.path = Path(path)
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

Создадим функцию обучения, чтобы запускать эксперимент с разными значениями `patience`. В конце загружаем лучшие веса из checkpoint.

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    with torch.no_grad():
        losses = [criterion(model(xb), yb).item() for xb, yb in loader]
    return float(np.mean(losses))


def train_with_early_stopping(patience, max_epochs=500):
    torch.manual_seed(42)
    train_ds, val_ds, feature_names, scaler = load_mpg()
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = MPGRegressor(len(feature_names))
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    checkpoint_path = f"best_model_patience_{patience}.pt"
    stopper = EarlyStopping(patience=patience, path=checkpoint_path)

    train_losses, val_losses = [], []
    stopped_epoch = max_epochs

    for epoch in range(max_epochs):
        model.train()
        batch_losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        val_loss = evaluate(model, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        stopper.step(val_loss, model)

        if stopper.should_stop:
            stopped_epoch = epoch + 1
            break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    final_train_loss = train_losses[-1]
    best_val_loss = stopper.best_loss
    return {
        "patience": patience,
        "epochs_ran": stopped_epoch,
        "final_train_loss": final_train_loss,
        "best_val_loss": best_val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses,
    }

Сравним три значения `patience`. Большее значение обычно дает модели больше времени, но обучение длится дольше.

In [ ]:
results = [train_with_early_stopping(p) for p in [3, 10, 25]]

summary = pd.DataFrame([
    {
        "patience": item["patience"],
        "epochs_ran": item["epochs_ran"],
        "final_train_loss": item["final_train_loss"],
        "best_val_loss": item["best_val_loss"],
    }
    for item in results
])
summary.round(2)

Построим кривые loss для каждого эксперимента и отметим вертикальной линией эпоху остановки. *

In [ ]:
plt.figure(figsize=(9, 5))

for item in results:
    epochs = np.arange(1, len(item["train_losses"]) + 1)
    plt.plot(epochs, item["train_losses"], linestyle="--", alpha=0.7, label=f"train p={item['patience']}")
    plt.plot(epochs, item["val_losses"], label=f"validation p={item['patience']}")
    plt.axvline(item["epochs_ran"], color="gray", alpha=0.25)

plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Early stopping comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

> Материал основан на курсе Jeff Heaton T81-558 и книге Maxim Lapan 'Deep Reinforcement Learning Hands-On', глава 3. Адаптировано для учебных целей.